# Task Runner Lifecycle
# 0. 介绍

**研究背景**：Agent 完成一个真实任务，通常要连续调用大模型和多个工具，并在每一步读取或修改外部系统。外层程序必须知道任务已经走到哪一步、哪些操作已经生效、失败后应该继续还是停止，以及什么结果才算真正完成。

**现存问题**：生产中常见的错误基线是只在内存里顺序执行步骤，遇到异常、超时或进程重启就从头再跑。写操作可能已经成功，只是返回结果在网络中丢失；此时盲目重试会重复建单、重复扣款或覆盖已有产物。已经完成的步骤也会因进度丢失而重复执行，甚至在最终产物尚未验证时就把任务标记为完成。

**解决方案**：本文档使用`持久化状态机 + Durable Execution（持久执行）+ 幂等与结果核对`：为任务和有副作用的操作分配稳定 ID，在每个阶段前后原子保存状态；遇到结果未知的超时，先查询外部环境确认操作是否已经生效，而不是立刻重放；对明确失败只限额重试当前阶段，重启后从 checkpoint 跳过已完成步骤，最后由环境验证结果决定能否进入完成状态。本 Notebook 将用同一份真实 API 计划进行对比：基线版本在超时后整单重跑并产生重复副作用；改进版本依靠持久状态、结果核对和断点恢复只生成一个通过验证的产物，从而直观看到可靠任务生命周期的关键不是“多重试几次”，而是“保存进度，并先弄清上一次操作是否已经发生”。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 定义模型提交计划的格式
Task Runner 必须先拿到结构明确的计划，才能按顺序推进任务。下面只要求大模型提交任务编号、计算结果和两个固定阶段，避免后续两种 Runner 因输入不同而失去可比性。

In [2]:
# 工具格式要求模型返回稳定的任务编号和计算结果
# stages 保存 Task Runner 后续必须依次推进的两个阶段
plan_tool = [{
    "type": "function",
    "function": {
        "name": "submit_plan",
        "description": "提交两阶段任务计划",
        "parameters": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string"},
                "result": {"type": "string"},
                "stages": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["create_job", "write_artifact"],
                    },
                },
            },
            "required": ["task_id", "result", "stages"],
            "additionalProperties": False,
        },
    },
}]

print(plan_tool[0]["function"]["name"])
print(plan_tool[0]["function"]["parameters"]["required"])

submit_plan
['task_id', 'result', 'stages']


输出显示大模型只能通过 `submit_plan` 提交三个必填字段。这个工具只统一计划格式，不执行任何阶段；下一步写出两种 Runner 共同处理的具体任务。

## 2.2 固定任务事实
为了让生命周期问题容易观察，任务只计算 `20 + 22`。完整流程仍有两个阶段：先在外部系统创建唯一作业记录，再把结果写入产物文件。

In [3]:
# task_id 在模型计划、外部记录和 Runner 状态之间保持不变
# stages 明确表示任务必须先建记录，再写出最终产物
task = {
    "task_id": "sum-20-22",
    "numbers": [20, 22],
    "stages": ["create_job", "write_artifact"],
    "artifact": "result.txt",
}

print(task)

{'task_id': 'sum-20-22', 'numbers': [20, 22], 'stages': ['create_job', 'write_artifact'], 'artifact': 'result.txt'}


输出固定了任务编号、输入数字、执行顺序和产物名称。后续基线版本与改进版本会完整复用这些事实；下一步把它们写成真实 API 可以读取的消息。

## 2.3 组装模型消息
大模型只负责计算结果并提交计划，不负责处理超时或恢复状态。下面把任务事实写进一条用户消息，让模型职责与 Task Runner 职责保持分离。

In [4]:
# system 消息要求模型只提交结构化计划
# user 消息提供唯一任务编号、数字和固定阶段
messages = [
    {
        "role": "system",
        "content": "计算结果，并调用 submit_plan 提交完整计划。",
    },
    {
        "role": "user",
        "content": (
            f"任务编号：{task['task_id']}；"
            f"计算：{task['numbers'][0]} + {task['numbers'][1]}；"
            f"阶段：{task['stages']}。"
        ),
    },
]

print(messages)

[{'role': 'system', 'content': '计算结果，并调用 submit_plan 提交完整计划。'}, {'role': 'user', 'content': "任务编号：sum-20-22；计算：20 + 22；阶段：['create_job', 'write_artifact']。"}]


输出显示真实 API 将看到两条消息，其中包含完整任务事实，但没有任何 Runner 状态或故障处理提示。模型输入已经固定；下一步为两条执行路径建立相同的空白外部系统。

## 2.4 建立相同的初始环境
基线版本与改进版本必须从相同状态开始。下面分别建立两个工作区，并在其中放入同样的空作业列表；后续只有 Runner 的处理方式不同。

In [5]:
from pathlib import Path
import shutil

# 实验目录固定在项目 artifacts 下，方便从头运行时重建
# 两个工作区都只写入同一份空作业列表
project_root = Path(find_dotenv()).parent
experiment_root = project_root / "artifacts" / "04_L_nanoTaskRunnerLifecycle_minimal"
baseline_workspace = experiment_root / "baseline"
fixed_workspace = experiment_root / "fixed"

for workspace in [baseline_workspace, fixed_workspace]:
    if workspace.exists():
        shutil.rmtree(workspace)
    workspace.mkdir(parents=True)
    (workspace / "jobs.json").write_text("[]", encoding="utf-8")

print("baseline：", (baseline_workspace / "jobs.json").read_text())
print("fixed：", (fixed_workspace / "jobs.json").read_text())

baseline： []
fixed： []


输出中的两份 `[]` 表示两个外部系统都没有作业记录，初始状态完全相同。环境差异已经排除；下一步规定两条路径共同使用的唯一成功标准。

## 2.5 定义成功标准
进程正常结束或模型提交计划都不等于任务成功。这个任务只有同时满足三项事实才算完成：外部系统只有一条作业记录、记录属于当前任务、产物内容等于正确结果。

In [6]:
# job_count 限制有副作用的建记录操作只能成功一次
# task_id 和 artifact_content 固定最终环境必须满足的事实
expected = {
    "job_count": 1,
    "task_id": task["task_id"],
    "artifact_content": "42",
}

print(expected)

{'job_count': 1, 'task_id': 'sum-20-22', 'artifact_content': '42'}


输出给出了后续所有实验共同使用的唯一标准。至此，计划格式、任务事实、模型消息、两套初始环境和成功条件都已固定；下一章将发送真实 API 请求并保存同一份模型计划。

# 3. 获取并验证 API 响应
## 3.1 发送真实 API 请求
现在把第 2 章固定的消息和计划格式交给真实大模型。这里强制模型调用 `submit_plan`，并记录等待时间；这次返回的同一份计划将同时交给后续两种 Runner。

In [7]:
from time import perf_counter

# 计时范围只包含这一次真实 API 请求
# 当前只有一个工具，required 要求模型通过它返回结构化计划
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=plan_tool,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = (perf_counter() - request_started) * 1000

print("真实 API 已返回计划")

真实 API 已返回计划


输出说明真实 API 已经返回，完整响应保存在 `response` 中。此时模型只提交了计划，外部系统还没有创建作业记录或结果文件；下一步读取这份结构化计划。

## 3.2 查看并保存模型计划
工具参数仍是 JSON 文本。下面把它转换成普通 Python 字典，并保存工具调用编号；后续基线版本和改进版本会直接复用这两个变量。

In [8]:
import json

# 第一条工具请求保存模型提交的完整计划
# 调用编号用于说明这份计划来自同一次真实响应
plan_call = response.choices[0].message.tool_calls[0]
model_plan = json.loads(plan_call.function.arguments)
plan_call_id = plan_call.id

print("tool：", plan_call.function.name)
print("plan：", model_plan)
print("call_id：", plan_call_id)

tool： submit_plan
plan： {'task_id': 'sum-20-22', 'result': '42', 'stages': ['create_job', 'write_artifact']}
call_id： call_3c162346e39d435c807bd387


输出展示了真实模型提交的任务编号、计算结果和两个执行阶段，并保留了本次工具调用编号。模型计划已经固定；下一步记录这次请求的来源、Token、成本、延迟和停止原因。

## 3.3 查看本次请求信息
模型响应还附带了运行信息。下面把这些数据整理成一个字典；API 没有直接返回金额，因此成本保留为 `None`，不使用猜测值代替。

In [9]:
# Token 数量直接读取真实 API 响应中的 usage
# stop_reason 只表示模型已经提交工具调用，不表示任务完成
request_info = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": response.usage.prompt_tokens,
    "output_tokens": response.usage.completion_tokens,
    "total_tokens": response.usage.total_tokens,
    "cost": None,
    "latency_ms": round(api_latency_ms, 2),
    "stop_reason": response.choices[0].finish_reason,
}

print(request_info)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 235, 'output_tokens': 139, 'total_tokens': 374, 'cost': None, 'latency_ms': 4545.5, 'stop_reason': 'tool_calls'}


输出记录了本次真实调用的 provider、model、Token、成本、延迟和停止原因。`tool_calls` 只说明模型正在等待外层程序执行计划，不代表任务已经成功；下一章将定义遇到超时就整项重跑的错误基线组件。

# 4. 定义基线组件
## 4.1 定义写入后超时的作业工具
生产系统最危险的超时发生在写入已经生效、但成功响应没有送回调用方之后。下面的工具先把作业保存到 `jobs.json`，第一次调用再抛出超时；因此超时只表示调用方不知道结果，不表示写入失败。

In [10]:
# 作业记录先写入外部文件，模拟服务端已经提交成功
# 第一条记录写入后才抛出超时，模拟成功响应在途中丢失
def create_job(workspace, task_id):
    jobs_path = workspace / "jobs.json"
    jobs = json.loads(jobs_path.read_text(encoding="utf-8"))
    job_id = f"JOB-{len(jobs) + 1}"
    jobs.append({"job_id": job_id, "task_id": task_id})
    jobs_path.write_text(json.dumps(jobs), encoding="utf-8")

    if len(jobs) == 1:
        raise TimeoutError("作业已写入，但成功响应丢失")

    return job_id


print("create_job 已定义")

create_job 已定义


输出说明作业工具已经定义，但两个工作区仍然为空。它的关键行为是“先写入、后超时”；下一步定义没有不确定结果的产物写入操作。

## 4.2 定义产物写入工具
第二个阶段只需要把计算结果写进 `result.txt`。这个工具不管理任务状态，只完成一次普通文件写入。

In [11]:
# 产物名称来自第 2 章固定的任务事实
# 写入内容直接使用真实模型提交的计算结果
def write_artifact(workspace, result):
    artifact_path = workspace / task["artifact"]
    artifact_path.write_text(result, encoding="utf-8")
    return artifact_path.name


print("write_artifact 已定义")

write_artifact 已定义


输出说明产物工具已经定义，但 `result.txt` 尚未生成。两个阶段的外部操作都已准备好；下一步用最常见的错误方式处理第一个阶段的超时。

## 4.3 定义盲目重试的旧 Runner
错误基线把任何超时都理解成“作业没有创建”，于是立即再次调用 `create_job`。它既不保存阶段状态，也不按任务编号查询外部系统；第二次调用成功返回后，就继续写产物并自报完成。

In [12]:
# 第一次超时后不核对外部状态，直接重复有副作用的写操作
# 第二次返回后继续写产物，并把进程结束当成任务完成
def baseline_runner(plan, workspace):
    try:
        job_id = create_job(workspace, plan["task_id"])
    except TimeoutError:
        job_id = create_job(workspace, plan["task_id"])

    artifact = write_artifact(workspace, plan["result"])
    return {"status": "done", "job_id": job_id, "artifact": artifact}


print("baseline_runner 已定义")

baseline_runner 已定义


输出说明错误基线已经定义，但尚未执行真实模型计划。它只有一条重试规则：超时就重复写入；下一章将运行它并直接查看 Runner 自报状态、外部作业记录和最终产物。

# 5. 展示基线故障
## 5.1 运行错误基线
现在把第 3 章的真实模型计划交给旧 Runner。第一次创建作业时，记录已经写入但调用方收到超时；旧 Runner 随即盲目重试，再继续写出结果文件。

In [13]:
# 输入直接复用真实模型提交的同一份 model_plan
# 所有副作用只发生在 baseline 工作区
baseline_result = baseline_runner(model_plan, baseline_workspace)

print(baseline_result)

{'status': 'done', 'job_id': 'JOB-2', 'artifact': 'result.txt'}


输出中的 `status` 是 `done`，说明旧 Runner 认为任务已经完成，并返回了第二次创建的 `JOB-2`。这只是进程内部结论；下一步直接读取外部系统，确认实际留下了多少条作业记录。

## 5.2 查看外部作业记录
超时发生在成功响应返回之前，所以不能只相信 Runner 的返回值。下面读取 `jobs.json`，查看服务端已经保存的真实状态。

In [14]:
# jobs.json 是 create_job 每次写入后的外部事实
# 这里完整展示记录数量、作业编号和稳定任务编号
baseline_jobs = json.loads(
    (baseline_workspace / "jobs.json").read_text(encoding="utf-8")
)

print("作业数量：", len(baseline_jobs))
print("作业记录：", baseline_jobs)

作业数量： 2
作业记录： [{'job_id': 'JOB-1', 'task_id': 'sum-20-22'}, {'job_id': 'JOB-2', 'task_id': 'sum-20-22'}]


输出显示同一个 `task_id` 对应 `JOB-1` 和 `JOB-2` 两条记录。第一次写入其实已经成功，第二条记录完全来自超时后的盲目重试；下一步确认重复记录是否同时影响了计算结果。

## 5.3 查看最终产物
重复作业不一定会让结果文件出错。下面单独读取 `result.txt`，把产物正确性与生命周期正确性分开观察。

In [15]:
# 产物路径来自模型计划执行后的 baseline 工作区
# 文件正文用于和第 2 章固定的正确结果比较
baseline_artifact = (
    baseline_workspace / task["artifact"]
).read_text(encoding="utf-8")

print(baseline_artifact)

42


输出是 `42`，说明真实模型的计算和结果写入都正确。但任务还要求外部系统只有一条作业记录；下一步使用第 2 章的完整标准判断基线结果。

## 5.4 判断基线结果
任务成功必须同时满足作业数量、任务编号和产物内容三项事实。下面逐项比较，避免用 Runner 自报的 `done` 代替真实环境结果。

In [16]:
# 三项检查直接对应第 2 章固定的 expected
# passed 只有在全部环境事实都正确时才会为 True
task_ids_match = True
for job in baseline_jobs:
    if job["task_id"] != expected["task_id"]:
        task_ids_match = False

baseline_grade = {
    "one_job": len(baseline_jobs) == expected["job_count"],
    "task_id_matches": task_ids_match,
    "artifact_matches": baseline_artifact == expected["artifact_content"],
}
baseline_grade["passed"] = all(baseline_grade.values())

print(baseline_grade)

{'one_job': False, 'task_id_matches': True, 'artifact_matches': True, 'passed': False}


输出中的 `passed` 为 `False`。模型结果、任务编号和产物都正确，唯一失败项是 `one_job`：旧 Runner 把结果未知的超时当成明确失败，重复执行了已经生效的写操作。下一章将定义能够保存进度并先核对外部状态的改进组件。

# 6. 定义改进组件
## 6.1 从磁盘加载任务状态
可靠的 Runner 不能只靠当前进程记住进度。下面优先读取 `runner_state.json`；首次运行时则建立一份空状态，其中稳定任务编号负责把模型计划、外部记录和 checkpoint 连接起来。

In [17]:
# 已有状态让新进程可以继续上一次运行
# 首次运行只创建任务编号、已完成阶段和作业编号三个字段
def load_state(workspace, task_id):
    state_path = workspace / "runner_state.json"

    if state_path.exists():
        return json.loads(state_path.read_text(encoding="utf-8"))

    return {
        "task_id": task_id,
        "completed_stages": [],
        "job_id": None,
    }


print("load_state 已定义")

load_state 已定义


输出说明状态加载器已经定义，但 fixed 工作区还没有状态文件。它负责回答“任务上次走到哪里”；下一步定义每个阶段完成后如何把答案持久保存。

## 6.2 原子保存任务状态
状态写到一半就中断，会留下无法恢复的 checkpoint。下面先完整写入临时文件，再用一次替换把它变成正式状态文件，使磁盘上始终只暴露完整版本。

In [18]:
# 临时文件先接收完整状态，避免直接改写正式 checkpoint
# replace 在写完后一次切换到新版本
def save_state(workspace, state):
    temporary_path = workspace / "runner_state.tmp"
    state_path = workspace / "runner_state.json"
    state_text = json.dumps(state, ensure_ascii=False)
    temporary_path.write_text(state_text, encoding="utf-8")
    temporary_path.replace(state_path)


print("save_state 已定义")

save_state 已定义


输出说明状态保存器已经定义，但尚未写入 checkpoint。加载与保存解决了进度丢失；下一步解决更关键的问题：超时后上一次写操作到底有没有生效。

## 6.3 按任务编号核对外部结果
结果未知时，正确动作不是立即重试，而是先查询外部系统。下面遍历现有作业，找到与稳定 `task_id` 对应的记录，并返回它已经分配到的作业编号。

In [19]:
# 外部 jobs.json 是判断写操作是否已经生效的事实来源
# 稳定 task_id 把超时前的请求与已保存记录对应起来
def find_job(workspace, task_id):
    jobs_path = workspace / "jobs.json"
    jobs = json.loads(jobs_path.read_text(encoding="utf-8"))

    for job in jobs:
        if job["task_id"] == task_id:
            return job["job_id"]


print("find_job 已定义")

find_job 已定义


输出说明结果核对器已经定义，但 fixed 工作区仍为空，因此还没有可返回的作业。下一步把 checkpoint 与核对动作组合成一个最小的持久化 Runner。

## 6.4 定义可恢复的 Task Runner
改进版 Runner 每次启动都先加载状态。创建作业遇到超时时，它先查找已经生效的记录；每完成一个阶段就保存 checkpoint，再次启动时直接跳过已完成阶段。

In [20]:
# completed_stages 决定本次运行需要执行还是跳过某个阶段
# 结果未知时先按 task_id 核对，找到记录后再保存进度
def durable_runner(plan, workspace):
    state = load_state(workspace, plan["task_id"])

    if "create_job" not in state["completed_stages"]:
        try:
            state["job_id"] = create_job(workspace, plan["task_id"])
        except TimeoutError:
            state["job_id"] = find_job(workspace, plan["task_id"])
        state["completed_stages"].append("create_job")
        save_state(workspace, state)

    if "write_artifact" not in state["completed_stages"]:
        write_artifact(workspace, plan["result"])
        state["completed_stages"].append("write_artifact")
        save_state(workspace, state)

    return {
        "status": "done",
        "job_id": state["job_id"],
        "artifact": task["artifact"],
    }


print("durable_runner 已定义")

durable_runner 已定义


输出说明改进组件已经定义，但尚未执行真实模型计划。它与基线版本使用相同工具，唯一变化是保存阶段进度，并把超时后的第一动作从“再次写入”改成“查询是否已经写入”；下一章将运行并恢复它。

# 7. 展示修复结果
## 7.1 运行可恢复的 Task Runner
现在把第 3 章保存的同一份真实模型计划交给改进版 Runner。创建作业仍会在写入后超时，但 Runner 会先按稳定任务编号核对外部记录，再继续第二个阶段。

In [21]:
# 输入继续复用 baseline 已经执行过的同一份 model_plan
# 所有改进路径的副作用只发生在 fixed 工作区
fixed_result = durable_runner(model_plan, fixed_workspace)

print(fixed_result)

{'status': 'done', 'job_id': 'JOB-1', 'artifact': 'result.txt'}


输出显示改进版返回 `JOB-1` 并自报完成。与基线的 `JOB-2` 不同，它在超时后找回了第一次已经创建的作业；下一步直接读取外部系统确认记录数量。

## 7.2 查看外部作业记录
Runner 返回值仍然不是最终事实。下面读取 fixed 工作区的 `jobs.json`，确认结果核对是否真的阻止了第二次写入。

In [22]:
# jobs.json 直接展示改进路径实际产生的外部副作用
# 记录中的 task_id 用于确认找回的是当前任务
fixed_jobs = json.loads(
    (fixed_workspace / "jobs.json").read_text(encoding="utf-8")
)

print("作业数量：", len(fixed_jobs))
print("作业记录：", fixed_jobs)

作业数量： 1
作业记录： [{'job_id': 'JOB-1', 'task_id': 'sum-20-22'}]


输出显示外部系统只有一条 `JOB-1`，任务编号也是 `sum-20-22`。超时仍然发生，但 Runner 没有重复写入；下一步查看它保存到磁盘的阶段进度。

## 7.3 查看持久化状态
外部作业只能说明副作用结果，checkpoint 才能说明 Runner 下次从哪里继续。下面重新从磁盘加载状态，不读取函数内部的临时变量。

In [23]:
# load_state 每次都从 fixed 工作区读取正式 checkpoint
# completed_stages 展示两个阶段是否已经持久完成
fixed_state = load_state(fixed_workspace, model_plan["task_id"])

print(fixed_state)

{'task_id': 'sum-20-22', 'completed_stages': ['create_job', 'write_artifact'], 'job_id': 'JOB-1'}


输出显示 checkpoint 保存了稳定任务编号、`JOB-1` 和两个已完成阶段。进度已经离开当前函数并写入磁盘；下一步确认最终产物没有因生命周期修复而改变。

## 7.4 查看最终产物
改进版减少了重复副作用，但仍必须完成原任务。下面读取 fixed 工作区的 `result.txt`，确认真实模型计算结果已经写出。

In [24]:
# 产物文件来自 checkpoint 中已经完成的 write_artifact 阶段
# 文件正文继续使用第 2 章的同一成功标准判断
fixed_artifact = (
    fixed_workspace / task["artifact"]
).read_text(encoding="utf-8")

print(fixed_artifact)

42


输出仍然是 `42`，说明改进版既避免了重复记录，也保留了正确产物。下一步再次调用同一个 Runner，模拟进程重新启动后从磁盘恢复。

## 7.5 从 checkpoint 再次运行
第二次调用不会继承上一次函数的局部变量，只能通过磁盘状态知道两个阶段已经完成。下面再次传入同一计划，观察 Runner 返回什么。

In [25]:
# 第二次运行仍从 load_state 开始读取磁盘 checkpoint
# 已完成阶段会被跳过，不再调用作业或产物写入工具
resumed_result = durable_runner(model_plan, fixed_workspace)

print(resumed_result)

{'status': 'done', 'job_id': 'JOB-1', 'artifact': 'result.txt'}


输出仍返回同一个 `JOB-1` 和同一个产物名称。Runner 声称恢复后没有重做阶段；下一步再次读取外部记录，用环境事实确认没有新增副作用。

## 7.6 查看恢复后的外部状态
如果 checkpoint 真正生效，第二次运行后作业数量应保持不变。下面重新读取 `jobs.json`，不复用第一次读取的列表。

In [26]:
# 再次读取磁盘，观察恢复调用之后的最新外部状态
# 数量保持为一说明已完成的 create_job 没有被重放
fixed_jobs_after_resume = json.loads(
    (fixed_workspace / "jobs.json").read_text(encoding="utf-8")
)

print("恢复后作业数量：", len(fixed_jobs_after_resume))
print("恢复后作业记录：", fixed_jobs_after_resume)

恢复后作业数量： 1
恢复后作业记录： [{'job_id': 'JOB-1', 'task_id': 'sum-20-22'}]


输出显示恢复后仍然只有 `JOB-1`。持久状态不仅修复了超时当次的重复写入，也让后续运行跳过已经完成的阶段；下一步使用与基线完全相同的标准判断最终结果。

## 7.7 判断修复结果
改进版不获得额外宽松条件。下面继续检查作业数量、任务编号和产物内容三项事实，并用同样的规则生成最终判断。

In [27]:
# 三项检查与 baseline_grade 使用完全相同的 expected
# 显式循环逐条确认恢复后的记录都属于当前任务
fixed_task_ids_match = True
for job in fixed_jobs_after_resume:
    if job["task_id"] != expected["task_id"]:
        fixed_task_ids_match = False

fixed_grade = {
    "one_job": len(fixed_jobs_after_resume) == expected["job_count"],
    "task_id_matches": fixed_task_ids_match,
    "artifact_matches": fixed_artifact == expected["artifact_content"],
}
fixed_grade["passed"] = all(fixed_grade.values())

print(fixed_grade)

{'one_job': True, 'task_id_matches': True, 'artifact_matches': True, 'passed': True}


输出中的四项结果全部为 `True`。模型、任务、工具和超时位置都没有变化；唯一变化是 Runner 保存了阶段状态，并在结果未知时先核对外部事实，因此只留下一个正确作业并能从 checkpoint 安全恢复。下一章将汇总两条路径的消融对照。

# 8. 汇总消融对照
## 8.1 确认共同实验条件
消融实验只有在其他条件相同时才有意义。下面记录两条路径共同使用的真实 provider、模型和计划；整个 Notebook 只请求一次模型，后续差异全部来自外层 Runner。

In [28]:
# provider、model 和计划都来自第 3 章的同一次真实响应
# baseline 与 fixed 直接复用同一个 model_plan 变量
shared_conditions = {
    "provider": request_info["provider"],
    "model": request_info["model"],
    "api_calls": 1,
    "model_plan": model_plan,
}

print(shared_conditions)

{'provider': 'openai', 'model': 'LongCat-2.0', 'api_calls': 1, 'model_plan': {'task_id': 'sum-20-22', 'result': '42', 'stages': ['create_job', 'write_artifact']}}


输出确认两条路径共享 `LongCat-2.0` 的同一份两阶段计划，API 调用数为 `1`。因此第 3 章记录的 Token、成本和延迟对两条路径完全相同；下一步只整理 Runner 造成的环境差异。

## 8.2 整理两条执行路径
下面用相同字段描述基线版本和改进版本：Runner 自报状态、外部作业数量、是否有 checkpoint、恢复后的作业数量、产物内容和最终任务判断。

In [29]:
# 两行数据分别读取已经运行完成的 baseline 与 fixed 环境事实
# 字段完全相同，便于直接观察唯一变量带来的结果差异
comparison_rows = [
    {
        "variant": "baseline",
        "runner_status": baseline_result["status"],
        "job_count": len(baseline_jobs),
        "checkpoint": (baseline_workspace / "runner_state.json").exists(),
        "resume_job_count": None,
        "artifact": baseline_artifact,
        "task_success": baseline_grade["passed"],
    },
    {
        "variant": "fixed",
        "runner_status": fixed_result["status"],
        "job_count": len(fixed_jobs),
        "checkpoint": (fixed_workspace / "runner_state.json").exists(),
        "resume_job_count": len(fixed_jobs_after_resume),
        "artifact": fixed_artifact,
        "task_success": fixed_grade["passed"],
    },
]

print("已整理执行路径：", len(comparison_rows))

已整理执行路径： 2


输出说明两条执行路径已经使用相同字段整理完成。数据仍保存在 `comparison_rows` 中；下一步逐行打印全部内容，不隐藏任何关键结果。

## 8.3 展示完整对照
现在逐行打印基线版本和改进版本。重点同时观察 `runner_status` 与 `task_success`：前者是程序自报，后者来自第 2 章固定的环境标准。

In [30]:
# 每次循环只展示一条完整执行路径
# 两行使用同一字段顺序，便于逐项比较
for row in comparison_rows:
    print(row)

{'variant': 'baseline', 'runner_status': 'done', 'job_count': 2, 'checkpoint': False, 'resume_job_count': None, 'artifact': '42', 'task_success': False}
{'variant': 'fixed', 'runner_status': 'done', 'job_count': 1, 'checkpoint': True, 'resume_job_count': 1, 'artifact': '42', 'task_success': True}


两种 Runner 都自报 `done`，产物也都是 `42`；但基线没有 checkpoint，留下两条作业并失败。改进版保存 checkpoint，首次和恢复后都只有一条作业，最终成功。下一步用三个状态变化概括机制效果。

## 8.4 总结机制效果
最后只比较加入持久状态与结果核对前后的关键变化。模型、计划、工具和故障位置保持不变，因此变化可以归因到 Task Runner Lifecycle。

In [31]:
# 外部作业数量直接反映超时后是否发生重复写入
# checkpoint 与任务成功状态连接了机制变化和最终结果
lifecycle_effect = {
    "job_count": f"{len(baseline_jobs)} -> {len(fixed_jobs_after_resume)}",
    "checkpoint": "False -> True",
    "task_success": f"{baseline_grade['passed']} -> {fixed_grade['passed']}",
}

for name, change in lifecycle_effect.items():
    print(name, "：", change)

job_count ： 2 -> 1
checkpoint ： False -> True
task_success ： False -> True


输出中的 `2 -> 1`、`False -> True` 和 `False -> True` 说明：模型从一开始就给出了正确计划，真正决定任务能否可靠完成的是外层 Runner 是否持久保存进度，并在结果未知时先核对外部事实。至此，本 Notebook 的消融对照结束。

## 8.5 拓展

### nano 版省略了什么

nano 版 Runner 只处理单任务、单故障点和内存 checkpoint，没有队列、并发、租约、心跳、取消、重试退避、幂等键、补偿事务、结果未知和跨版本恢复。生产生命周期还需把 queued/running/waiting/failed/succeeded 等状态与外部副作用对齐，避免进程状态冒充任务事实。

### 延伸阅读


1. 2025, [Anthropic, Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)：跨会话增量执行、初始化与交接。
2. 2026, [LangGraph, Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution)：持久状态、确定性恢复与副作用封装。
3. 2026, [OpenAI, Unrolling the Codex agent loop](https://openai.com/index/unrolling-the-codex-agent-loop)：循环状态、工具执行和性能边界。